# STT Processing Workflow with Groq

Этот ноутбук содержит воркфлоу для обработки текста, полученного из STT (Speech-to-Text).
1. **Первый проход**: LLM (Groq) принимает расшифровку STT (Input) и разбивает текст, формируя структурированный JSON с раскадровкой ответов на вопросы.
2. **Цикл оценок**: Выполняется 6 независимых вызовов LLM для оценки каждого из ответов по критериям (q1-q6).
3. **Объединение (Aggregation)**: Все оценки алгоритмически собираются в единый финальный JSON документ.

In [ ]:
# Устанавливаем необходимые зависимости
!pip install -q groq python-dotenv

In [103]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

# Загружаем переменные окружения из .env (предполагаем наличие GROQ_API_KEY)
load_dotenv()

# Инициализируем клиента Groq
# Обязательно добавьте Ваш токен в переменную окружения $GROQ_API_KEY
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Выбираем модель (llama3-70b-8192 отлично подходит для сложных промтов)
MODEL_NAME = os.environ.get("GROQ_MODEL", "llama3-70b-8192")

def read_prompt(filename):
    """Вспомогательная функция для чтения файлов промптов."""
    path = os.path.join("prompts", filename)
    with open(path, "r", encoding="utf-8") as f:
         return f.read()
    
def read_stt_txt(filename):
    """Вспомогательная функция для чтения текстовых файлов с результатами STT."""
    with open(filename, "r", encoding="utf-8") as f:
         return f.read()

def call_groq(prompt_text, system_message="Вы полезный HR ассистент.", require_json=True, max_tokens=None, max_retries=3):
    """Делает вызов к Groq API и возвращает распарсенный JSON."""
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt_text}
    ]
    
    # Задаем response_format для гарантированного возврата JSON
    response_args = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": 0.1,
        "max_tokens": max_tokens
    }
    if require_json:
        response_args["response_format"] = {"type": "json_object"}
    
    response = client.chat.completions.create(**response_args)
    content = response.choices[0].message.content
    
    if require_json:
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            print(f"Ошибка парсинга JSON:\n{content}")
            return None
    return content

In [111]:
# Тестовый пример STT (замените на чтение из вашего реального источника/файла)
stt_input = read_stt_txt("input.txt")
# --- ШАГ 1: Первый проход LLM (Парсинг текста на раскадровку вопросов) ---
print("Запуск первого прохода LLM (Парсер текста)...")

# Мы используем главный промпт-парсер как системный промпт
parser_system_prompt = read_prompt("prompt_main_parser.txt")

# Пользовательский ввод — это только текст STT
parser_user_prompt = stt_input.strip()

parsed_stt_json = call_groq(
    prompt_text=parser_user_prompt, 
    system_message=parser_system_prompt,
    require_json=True, # Включаем обратно строгий JSON
    max_tokens=8000, # Добавляем больше токенов для длинного текста
    max_retries=3 # Добавляем повторные попытки на случай ошибок парсинга
)

print("\nРезультат первого прохода (Раскадровка STT текста):")
print(json.dumps(parsed_stt_json, indent=4, ensure_ascii=False))

Запуск первого прохода LLM (Парсер текста)...

Результат первого прохода (Раскадровка STT текста):
{
    "questions": {
        "q1_text": "why I apply to inVision U, because I feel I need this type environment, not only for study like passive, but place where people do real things, make project, discuss, try, fail, improve, and I think for me this is important now because I learn better when I am inside serious people and serious process, not when I just sit and listen, and also I think when around you people who want more from life, you also start ask more from yourself, and I want this",
        "q2_text": "about program, I think more leadership and entrepreneurship part, because I like when situation is not clear yet, maybe little chaos, and somebody need start, like okay what is problem, what first, who can do what, how we make first version, because I am not person who know everything, no, but I feel interest when from confusion you make some movement, some structure, and I want 

In [112]:
# --- ШАГ 2: Цикл 6 LLM проходов для постановки оценки каждому ответу (q1-q6) ---

evaluations = {}

if parsed_stt_json and isinstance(parsed_stt_json, dict):
    print("Запуск цикла оценок (6 независимых LLM проходов)...")
    
    # Достаем блок с ответами из распарсенного JSON (как указано в prompt_main_parser.txt)
    questions_data = parsed_stt_json.get("questions", {})
    
    # Итерируемся от 1 до 6
    for i in range(1, 7):
        q_key = f"q{i}"
        q_text_key = f"q{i}_text"
        prompt_filename = f"q{i}_prompt.txt"
        
        # Получаем ответ кандидата
        candidate_answer = questions_data.get(q_text_key, "") or ""
        print(f"[{q_key}] Оцениваем ответ: {candidate_answer[:50]}...")
        
        # Читаем локальный промпт оценки (он будет system_message)
        try:
            eval_system_prompt = read_prompt(prompt_filename)
        except FileNotFoundError:
            print(f"Промпт {prompt_filename} не найден в папке 'prompts', пропускаем...")
            continue
            
        # Получаем оценку
        eval_result = call_groq(
            prompt_text=candidate_answer, 
            system_message=eval_system_prompt,
            require_json=True
        )
        
        # Сохраняем оценку в общий словарь
        evaluations[q_key] = eval_result
        
else:
    print("Ошибка на Шаге 1: Распарсенный JSON пуст или парсинг не удался.")

print("\nОценки по вопросам успешно проставлены (или цикл завершен).")

Запуск цикла оценок (6 независимых LLM проходов)...
[q1] Оцениваем ответ: why I apply to inVision U, because I feel I need t...
[q2] Оцениваем ответ: about program, I think more leadership and entrepr...
[q3] Оцениваем ответ: major challenge for me was beginning in new academ...
[q4] Оцениваем ответ: long term I want build things which help other peo...
[q5] Оцениваем ответ: for me leadership is not loud talking or looking s...
[q6] Оцениваем ответ: yes my family support me, maybe they cannot explai...

Оценки по вопросам успешно проставлены (или цикл завершен).


In [113]:
# --- ШАГ 3: Алгоритмическое объединение данных в единый финальный JSON --- 

def extract_score(evaluations, question_key, metric_name):
    """Вспомогательная функция для безопасного извлечения оценок из JSON LLM"""
    try:
        if question_key in evaluations and evaluations[question_key]:
            # Ищем нужную метрику в массиве "scores", если такой формат
            if "scores" in evaluations[question_key]:
                for item in evaluations[question_key]["scores"]:
                    if isinstance(item, dict) and item.get("metric_name") == metric_name:
                        return float(item.get("score", 0))
    except Exception:
        pass
    return 0.0

# 1. Извлекаем все баллы по метрикам и вопросам
scores = {
    "q1_motivation": extract_score(evaluations, "q1", "motivation"),
    "q1_planning": extract_score(evaluations, "q1", "planning"),
    
    "q2_motivation": extract_score(evaluations, "q2", "motivation"),
    "q2_planning": extract_score(evaluations, "q2", "planning"),
    
    "q3_resilience": extract_score(evaluations, "q3", "resilience"),
    "q3_leadership": extract_score(evaluations, "q3", "leadership"),
    "q3_values": extract_score(evaluations, "q3", "values"),
    
    "q4_planning": extract_score(evaluations, "q4", "planning"),
    "q4_motivation": extract_score(evaluations, "q4", "motivation"),
    
    "q5_leadership": extract_score(evaluations, "q5", "leadership"),
    "q5_values": extract_score(evaluations, "q5", "values"),
    
    "q6_social_support": extract_score(evaluations, "q6", "social_support"),
    "q6_resilience": extract_score(evaluations, "q6", "resilience"),
    "q6_motivation": extract_score(evaluations, "q6", "motivation"),
}

# 2. Считаем агрегированные метрики по формулам
Agg_M = (0.35 * scores["q1_motivation"]) + (0.20 * scores["q2_motivation"]) + (0.35 * scores["q4_motivation"]) + (0.10 * scores["q6_motivation"])
Agg_P = (0.15 * scores["q1_planning"]) + (0.35 * scores["q2_planning"]) + (0.50 * scores["q4_planning"])
Agg_R = (0.80 * scores["q3_resilience"]) + (0.20 * scores["q6_resilience"])
Agg_L = (0.30 * scores["q3_leadership"]) + (0.70 * scores["q5_leadership"])
Agg_V = (0.40 * scores["q3_values"]) + (0.60 * scores["q5_values"])
Agg_S = 1.0 * scores["q6_social_support"]

# 3. Считаем глобальные индексы
LeadershipIndex = (0.35 * Agg_L) + (0.20 * Agg_R) + (0.20 * Agg_P) + (0.15 * Agg_M) + (0.10 * Agg_V)
AdmissionsPotential = (0.25 * Agg_L) + (0.20 * Agg_P) + (0.20 * Agg_M) + (0.20 * Agg_R) + (0.10 * Agg_V) + (0.05 * Agg_S)


final_combined_output = {
    "workflow_status": "success",
    "stt_length": len(stt_input) if stt_input else 0,
    "candidate_breakdown": parsed_stt_json.get("questions", parsed_stt_json) if isinstance(parsed_stt_json, dict) else parsed_stt_json,
    "llm_evaluations": evaluations,
    "aggregated_metrics": {
        "Motivation": round(Agg_M, 2),
        "Planning": round(Agg_P, 2),
        "Resilience": round(Agg_R, 2),
        "Leadership": round(Agg_L, 2),
        "Values": round(Agg_V, 2),
        "Social_Support": round(Agg_S, 2)
    },
    "global_score": {
        "LeadershipIndex": round(LeadershipIndex, 2),
        "AdmissionsPotential": round(AdmissionsPotential, 2)
    }
}

# Формируем итоговую JSON-строку
final_json_str = json.dumps(final_combined_output, indent=4, ensure_ascii=False)

print("=== ФИНАЛЬНЫЙ СТРУКТУРИРОВАННЫЙ JSON ОБЪЕКТ ===")
print(final_json_str)

# Сохраняем в файл 
with open("final_interview_evaluation.json", "w", encoding="utf-8") as f:
    f.write(final_json_str)

print("Результат сохранен в 'final_interview_evaluation.json'")

=== ФИНАЛЬНЫЙ СТРУКТУРИРОВАННЫЙ JSON ОБЪЕКТ ===
{
    "workflow_status": "success",
    "stt_length": 4066,
    "candidate_breakdown": {
        "q1_text": "why I apply to inVision U, because I feel I need this type environment, not only for study like passive, but place where people do real things, make project, discuss, try, fail, improve, and I think for me this is important now because I learn better when I am inside serious people and serious process, not when I just sit and listen, and also I think when around you people who want more from life, you also start ask more from yourself, and I want this",
        "q2_text": "about program, I think more leadership and entrepreneurship part, because I like when situation is not clear yet, maybe little chaos, and somebody need start, like okay what is problem, what first, who can do what, how we make first version, because I am not person who know everything, no, but I feel interest when from confusion you make some movement, some struc